# LeetCode 1310 — XOR Queries of a Subarray bằng Python

Link đề: https://leetcode.com/problems/xor-queries-of-a-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def xorQueries(self, arr: list[int], queries: list[list[int]]) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Hiểu phép **XOR** (`^`) trên bit và các tính chất vàng: `a ^ 0 = a`, `a ^ a = 0`, giao hoán, kết hợp.
2. Nhận ra XOR có **phép ngược là chính nó** ⇒ áp dụng được kỹ thuật **prefix (tiền tố)** giống prefix sum.
3. Trả lời mỗi truy vấn đoạn `[l, r]` trong `O(1)` bằng `pre[r+1] ^ pre[l]` sau `O(n)` tiền xử lý.
4. Biết thêm bước trung gian **chia khối (sqrt decomposition)** và cách viết gọn bằng `itertools.accumulate` + `operator.xor`.
5. Tránh lỗi: lệch chỉ số tiền tố, dùng `-` thay `^`, nhầm `^` là luỹ thừa, chỉ số `-1` "im lặng" trong Python.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `arr` và mảng truy vấn `queries`, trong đó `queries[i] = [left_i, right_i]`. Với mỗi truy vấn, tính `arr[left] XOR arr[left+1] XOR … XOR arr[right]`. Trả về mảng các kết quả theo thứ tự truy vấn.

**Ví dụ 1:** `arr = [1,3,4,8], queries = [[0,1],[1,2],[0,3],[3,3]]` → `[2,7,14,8]`

```
1 = 0001, 3 = 0011, 4 = 0100, 8 = 1000
[0,1] = 1 xor 3 = 2
[1,2] = 3 xor 4 = 7
[0,3] = 1 xor 3 xor 4 xor 8 = 14
[3,3] = 8
```

**Ví dụ 2:** `arr = [4,8,2,10], queries = [[2,3],[1,3],[0,0],[0,3]]` → `[8,0,4,4]`

**Ràng buộc:** `1 <= arr.length, queries.length <= 3·10^4`, `1 <= arr[i] <= 10^9`, `queries[i].length == 2`, `0 <= left_i <= right_i < arr.length`.

In [1]:
import operator
import random
import time
from functools import reduce
from itertools import accumulate
from typing import List

examples = [
    (([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]), [2, 7, 14, 8]),
    (([4, 8, 2, 10], [[2, 3], [1, 3], [0, 0], [0, 3]]), [8, 0, 4, 4]),
]
for (arr, queries), want in examples:
    print(f"arr={arr}, queries={queries} -> mong đợi {want}")

arr=[1, 3, 4, 8], queries=[[0, 1], [1, 2], [0, 3], [3, 3]] -> mong đợi [2, 7, 14, 8]
arr=[4, 8, 2, 10], queries=[[2, 3], [1, 3], [0, 0], [0, 3]] -> mong đợi [8, 0, 4, 4]


## 3. Kiến thức nền

### 3.1 Phép XOR trên bit

XOR ("hoặc loại trừ") so từng cặp bit: **khác nhau → 1, giống nhau → 0**. Trong Python là toán tử `^`. Dùng `format(x, '04b')` để xem dạng nhị phân.

In [2]:
a, b = 3, 4
print(f"{a:04b}  ({a})")
print(f"{b:04b}  ({b})")
print(f"{a ^ b:04b}  ({a ^ b})  <- 3 ^ 4")
print("bảng chân trị:", [(x, y, x ^ y) for x in (0, 1) for y in (0, 1)])

0011  (3)
0100  (4)
0111  (7)  <- 3 ^ 4
bảng chân trị: [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 0)]


### 3.2 Bốn tính chất vàng của XOR

1. `a ^ 0 = a` (0 là phần tử trung hoà).
2. `a ^ a = 0` (mỗi số là **nghịch đảo của chính nó**).
3. Giao hoán: `a ^ b = b ^ a`.
4. Kết hợp: `(a ^ b) ^ c = a ^ (b ^ c)`.

Hệ quả: `a ^ b ^ a = b` — XOR thêm một số hai lần thì "huỷ" nó.

In [3]:
rng = random.Random(0)
for _ in range(5):
    a, b, c = (rng.randint(0, 10**9) for _ in range(3))
    assert a ^ 0 == a and a ^ a == 0 and a ^ b == b ^ a and (a ^ b) ^ c == a ^ (b ^ c)
print("4 tính chất đúng trên mẫu ngẫu nhiên")
print("7 ^ 12 ^ 7 =", 7 ^ 12 ^ 7, "(7 bị huỷ)")

4 tính chất đúng trên mẫu ngẫu nhiên
7 ^ 12 ^ 7 = 12 (7 bị huỷ)


### 3.3 Nhắc lại prefix sum

Với tổng: `pre[i] = arr[0] + … + arr[i-1]` (`pre[0] = 0`). Tổng đoạn `[l, r]` = `pre[r+1] − pre[l]`. Phép trừ "huỷ" phần đầu `arr[0..l-1]`. Với XOR, phép "huỷ" chính là `^` (tính chất 2).

In [4]:
arr = [1, 3, 4, 8]
pre_sum = [0] + list(accumulate(arr))
print("prefix sum:", pre_sum, "| tổng [1,2] =", pre_sum[3] - pre_sum[1], "=", 3 + 4)

prefix sum: [0, 1, 4, 8, 16] | tổng [1,2] = 7 = 7


### 3.4 `functools.reduce`, `itertools.accumulate`, `operator.xor`

- `reduce(operator.xor, seq, 0)`: XOR toàn bộ dãy.
- `accumulate(seq, operator.xor, initial=0)`: dãy XOR tiền tố (có phần tử 0 ở đầu).

In [5]:
arr = [1, 3, 4, 8]
print("XOR cả mảng:", reduce(operator.xor, arr, 0))
print("prefix XOR :", list(accumulate(arr, operator.xor, initial=0)))

XOR cả mảng: 14
prefix XOR : [0, 1, 2, 6, 14]


### 3.5 Cảnh giác: `^` không phải luỹ thừa

Trong toán học hay viết `2^3 = 8`, nhưng trong Python `2 ^ 3` là XOR = `1`. Luỹ thừa là `**`.

In [6]:
print("2 ^ 3  =", 2 ^ 3, "(XOR)")
print("2 ** 3 =", 2 ** 3, "(luỹ thừa)")

2 ^ 3  = 1 (XOR)
2 ** 3 = 8 (luỹ thừa)


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `arr = [1, 3, 4, 8]`. Dựng tiền tố XOR `pre[i] = arr[0] ^ … ^ arr[i-1]`:

| i | pre[i] | cách tính |
|---|---|---|
| 0 | 0 | rỗng |
| 1 | 1 | 1 |
| 2 | 2 | 1 ^ 3 |
| 3 | 6 | 1 ^ 3 ^ 4 |
| 4 | 14 | 1 ^ 3 ^ 4 ^ 8 |

Truy vấn `[1, 2]`: `pre[3] ^ pre[1] = (1^3^4) ^ (1) = 3 ^ 4 = 7` ✔ — số `1` xuất hiện hai lần nên tự huỷ.

Truy vấn `[3, 3]`: `pre[4] ^ pre[3] = 14 ^ 6 = 8` ✔.

In [7]:
arr = [1, 3, 4, 8]
pre = [0]
for x in arr:
    pre.append(pre[-1] ^ x)
print("pre =", pre)
for l, r in [[0, 1], [1, 2], [0, 3], [3, 3]]:
    print(f"[{l},{r}] = pre[{r + 1}] ^ pre[{l}] = {pre[r + 1]} ^ {pre[l]} = {pre[r + 1] ^ pre[l]}")

pre = [0, 1, 2, 6, 14]
[0,1] = pre[2] ^ pre[0] = 2 ^ 0 = 2
[1,2] = pre[3] ^ pre[1] = 6 ^ 1 = 7
[0,3] = pre[4] ^ pre[0] = 14 ^ 0 = 14
[3,3] = pre[4] ^ pre[3] = 14 ^ 6 = 8


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi truy vấn, XOR lần lượt các phần tử từ `left` đến `right`.

In [8]:
def xor_queries_brute(arr, queries):
    res = []
    for l, r in queries:
        x = 0
        for i in range(l, r + 1):
            x ^= arr[i]
        res.append(x)
    return res

for (arr, queries), want in examples:
    print(xor_queries_brute(arr, queries), "| mong đợi", want)

[2, 7, 14, 8] | mong đợi [2, 7, 14, 8]
[8, 0, 4, 4] | mong đợi [8, 0, 4, 4]


**Độ phức tạp:** gọi `n = len(arr)`, `q = len(queries)`.
- Time `O(q · n)` trường hợp xấu (mọi truy vấn là cả mảng): `3·10^4 · 3·10^4 = 9·10^8` ⇒ **quá chậm**.
- Space `O(1)` ngoài kết quả.

**Nút thắt:** các truy vấn chồng lấn nhau nhưng ta tính lại từ đầu mỗi lần.

In [9]:
rng = random.Random(1)
n = 3000
arr_t = [rng.randint(1, 10**9) for _ in range(n)]
qs_t = [[0, n - 1]] * 3000
t0 = time.perf_counter()
xor_queries_brute(arr_t, qs_t)
print(f"brute n=q=3000, mọi truy vấn cả mảng: {time.perf_counter() - t0:.2f}s (n=q=3·10^4 sẽ chậm gấp ~100 lần)")

brute n=q=3000, mọi truy vấn cả mảng: 0.53s (n=q=3·10^4 sẽ chậm gấp ~100 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `P(k) = arr[0] ^ arr[1] ^ … ^ arr[k-1]` (`P(0) = 0`). Khi đó

```
P(r+1) ^ P(l) = (arr[0] ^ … ^ arr[l-1] ^ arr[l] ^ … ^ arr[r]) ^ (arr[0] ^ … ^ arr[l-1])
             = arr[l] ^ … ^ arr[r]          (vì mỗi arr[i], i < l, xuất hiện 2 lần ⇒ huỷ)
```

**Quy tắc:** *nếu* một phép toán có tính kết hợp và mỗi phần tử có **nghịch đảo** (tổng: `−x`; XOR: chính `x`) *thì* có thể dùng tiền tố để trả lời truy vấn đoạn trong `O(1)`.

(Ngược lại, `max`/`min` **không** có nghịch đảo ⇒ không dùng được tiền tố kiểu này; phải dùng sparse table/segment tree.)

**Invariant khi dựng:** sau bước `i`, `pre[i+1]` là XOR của `arr[0..i]`.

In [10]:
# Kiểm chứng đẳng thức trên mẫu ngẫu nhiên
rng = random.Random(2)
arr = [rng.randint(1, 100) for _ in range(12)]
pre = list(accumulate(arr, operator.xor, initial=0))
for _ in range(5):
    l = rng.randrange(12); r = rng.randrange(l, 12)
    direct = reduce(operator.xor, arr[l:r + 1], 0)
    print(f"[{l},{r}]: trực tiếp={direct:3d}  pre[r+1]^pre[l]={pre[r + 1] ^ pre[l]:3d}")
    assert direct == pre[r + 1] ^ pre[l]

[0,9]: trực tiếp=120  pre[r+1]^pre[l]=120
[10,10]: trực tiếp= 28  pre[r+1]^pre[l]= 28
[6,11]: trực tiếp= 67  pre[r+1]^pre[l]= 67
[6,11]: trực tiếp= 67  pre[r+1]^pre[l]= 67
[8,10]: trực tiếp=115  pre[r+1]^pre[l]=115


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chia khối (sqrt decomposition) — bước trung gian

Chia mảng thành các khối kích thước `B ≈ √n`, tính trước XOR của mỗi khối. Một truy vấn `[l, r]` = phần lẻ đầu + các khối trọn vẹn + phần lẻ cuối ⇒ `O(√n)` mỗi truy vấn. Không tối ưu bằng tiền tố cho bài này, nhưng là kỹ thuật hữu ích khi có **cập nhật** phần tử (cập nhật một khối `O(√n)`... hoặc `O(1)` với XOR: `block[b] ^= old ^ new`).

In [11]:
def xor_queries_blocks(arr, queries):
    n = len(arr)
    B = max(1, int(n ** 0.5))
    blocks = [0] * ((n + B - 1) // B)
    for i, x in enumerate(arr):
        blocks[i // B] ^= x
    res = []
    for l, r in queries:
        x = 0
        i = l
        while i <= r:
            if i % B == 0 and i + B - 1 <= r:   # cả khối nằm trong [l, r]
                x ^= blocks[i // B]
                i += B
            else:
                x ^= arr[i]
                i += 1
        res.append(x)
    return res

for (arr, queries), want in examples:
    print(xor_queries_blocks(arr, queries), "| mong đợi", want)

[2, 7, 14, 8] | mong đợi [2, 7, 14, 8]
[8, 0, 4, 4] | mong đợi [8, 0, 4, 4]


Code nộp LeetCode (cách 2 — qua được nhưng chậm hơn cách 3):

In [12]:
class SolutionBlocks:
    def xorQueries(self, arr: List[int], queries: List[List[int]]) -> List[int]:
        n = len(arr)
        B = max(1, int(n ** 0.5))
        blocks = [0] * ((n + B - 1) // B)
        for i, x in enumerate(arr):
            blocks[i // B] ^= x
        res = []
        for l, r in queries:
            x, i = 0, l
            while i <= r:
                if i % B == 0 and i + B - 1 <= r:
                    x ^= blocks[i // B]
                    i += B
                else:
                    x ^= arr[i]
                    i += 1
            res.append(x)
        return res

print(SolutionBlocks().xorQueries([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]))

[2, 7, 14, 8]


**Độ phức tạp cách 2:** Time `O(n + q·√n)` (mỗi truy vấn tối đa `2B` phần tử lẻ + `n/B` khối), Space `O(√n)`.

### 7.2 Cách 3 — Prefix XOR

1. `pre = [0] * (n + 1)`; `pre[i+1] = pre[i] ^ arr[i]`.
2. Mỗi truy vấn `[l, r]` ⇒ `pre[r+1] ^ pre[l]`.

In [13]:
def xor_queries_prefix(arr, queries):
    n = len(arr)
    pre = [0] * (n + 1)
    for i, x in enumerate(arr):
        pre[i + 1] = pre[i] ^ x
    return [pre[r + 1] ^ pre[l] for l, r in queries]

for (arr, queries), want in examples:
    print(xor_queries_prefix(arr, queries), "| mong đợi", want)

[2, 7, 14, 8] | mong đợi [2, 7, 14, 8]
[8, 0, 4, 4] | mong đợi [8, 0, 4, 4]


Code nộp LeetCode (cách 3):

In [14]:
class SolutionPrefix:
    def xorQueries(self, arr: List[int], queries: List[List[int]]) -> List[int]:
        pre = [0] * (len(arr) + 1)
        for i, x in enumerate(arr):
            pre[i + 1] = pre[i] ^ x
        return [pre[r + 1] ^ pre[l] for l, r in queries]

print(SolutionPrefix().xorQueries([4, 8, 2, 10], [[2, 3], [1, 3], [0, 0], [0, 3]]))

[8, 0, 4, 4]


**Độ phức tạp cách 3:** Time `O(n + q)`, Space `O(n)` cho mảng `pre`.

### 7.3 Cách 4 — `accumulate` gọn một dòng / tiền tố tại chỗ

`accumulate(arr, operator.xor, initial=0)` dựng ngay mảng `pre` bằng code C bên trong ⇒ ngắn và nhanh. Nếu được phép sửa input, có thể biến `arr` thành tiền tố tại chỗ (`arr[i] ^= arr[i-1]`) để tiết kiệm bộ nhớ; khi đó truy vấn `[l, r]` = `arr[r] ^ (arr[l-1] if l > 0 else 0)`.

In [15]:
def xor_queries_accumulate(arr, queries):
    pre = list(accumulate(arr, operator.xor, initial=0))
    return [pre[r + 1] ^ pre[l] for l, r in queries]

def xor_queries_inplace(arr, queries):
    for i in range(1, len(arr)):
        arr[i] ^= arr[i - 1]                 # arr[i] = XOR của arr[0..i] gốc
    return [arr[r] ^ (arr[l - 1] if l > 0 else 0) for l, r in queries]

for (arr, queries), want in examples:
    print(xor_queries_accumulate(arr, queries), xor_queries_inplace(list(arr), queries), "| mong đợi", want)

[2, 7, 14, 8] [2, 7, 14, 8] | mong đợi [2, 7, 14, 8]
[8, 0, 4, 4] [8, 0, 4, 4] | mong đợi [8, 0, 4, 4]


Code nộp LeetCode (cách 4):

In [16]:
class SolutionAccumulate:
    def xorQueries(self, arr: List[int], queries: List[List[int]]) -> List[int]:
        pre = list(accumulate(arr, operator.xor, initial=0))
        return [pre[r + 1] ^ pre[l] for l, r in queries]

print(SolutionAccumulate().xorQueries([1, 3, 4, 8], [[0, 3]]))

[14]


**Độ phức tạp cách 4:** Time `O(n + q)`, Space `O(n)` (bản `accumulate`) hoặc `O(1)` phụ (bản tại chỗ, nhưng sửa input).

In [17]:
rng = random.Random(3)
n = q = 3 * 10**4
arr_big = [rng.randint(1, 10**9) for _ in range(n)]
qs_big = []
for _ in range(q):
    l = rng.randrange(n); r = rng.randrange(l, n)
    qs_big.append([l, r])
for name, f in [("blocks", xor_queries_blocks), ("prefix", xor_queries_prefix), ("accumulate", xor_queries_accumulate)]:
    t0 = time.perf_counter()
    res = f(arr_big, qs_big)
    print(f"{name:10s}: {time.perf_counter() - t0:.4f}s")

blocks    : 0.7221s
prefix    : 0.0059s
accumulate: 0.0033s


## 8. TRACE từng bước

Trace cách 3 trên Ví dụ 2: `arr = [4, 8, 2, 10]`.

**Pha 1 — dựng tiền tố:**

| i | arr[i] (nhị phân) | pre[i] | pre[i+1] = pre[i] ^ arr[i] |
|---|---|---|---|
| 0 | 4 (0100) | 0 (0000) | 4 (0100) |
| 1 | 8 (1000) | 4 (0100) | 12 (1100) |
| 2 | 2 (0010) | 12 (1100) | 14 (1110) |
| 3 | 10 (1010) | 14 (1110) | 4 (0100) |

**Pha 2 — truy vấn:**

| [l, r] | pre[r+1] ^ pre[l] | kết quả |
|---|---|---|
| [2, 3] | 4 ^ 12 | 8 |
| [1, 3] | 4 ^ 4 | 0 |
| [0, 0] | 4 ^ 0 | 4 |
| [0, 3] | 4 ^ 0 | 4 |

In [18]:
def trace_prefix(arr, queries):
    n = len(arr)
    pre = [0] * (n + 1)
    print("Pha 1: dựng tiền tố")
    for i, x in enumerate(arr):
        pre[i + 1] = pre[i] ^ x
        print(f"  i={i}: pre[{i}]={pre[i]:2d} ({pre[i]:04b}) ^ arr[{i}]={x:2d} ({x:04b}) -> pre[{i + 1}]={pre[i + 1]:2d} ({pre[i + 1]:04b})")
    print("Pha 2: trả lời truy vấn")
    res = []
    for l, r in queries:
        v = pre[r + 1] ^ pre[l]
        print(f"  [{l},{r}]: pre[{r + 1}]={pre[r + 1]} ^ pre[{l}]={pre[l]} = {v}")
        res.append(v)
    print("=>", res)
    return res

trace_prefix([4, 8, 2, 10], [[2, 3], [1, 3], [0, 0], [0, 3]])

Pha 1: dựng tiền tố
  i=0: pre[0]= 0 (0000) ^ arr[0]= 4 (0100) -> pre[1]= 4 (0100)
  i=1: pre[1]= 4 (0100) ^ arr[1]= 8 (1000) -> pre[2]=12 (1100)
  i=2: pre[2]=12 (1100) ^ arr[2]= 2 (0010) -> pre[3]=14 (1110)
  i=3: pre[3]=14 (1110) ^ arr[3]=10 (1010) -> pre[4]= 4 (0100)
Pha 2: trả lời truy vấn
  [2,3]: pre[4]=4 ^ pre[2]=12 = 8
  [1,3]: pre[4]=4 ^ pre[1]=4 = 0
  [0,0]: pre[1]=4 ^ pre[0]=0 = 4
  [0,3]: pre[4]=4 ^ pre[0]=0 = 4
=> [8, 0, 4, 4]


[8, 0, 4, 4]

In [19]:
trace_prefix([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]])

Pha 1: dựng tiền tố
  i=0: pre[0]= 0 (0000) ^ arr[0]= 1 (0001) -> pre[1]= 1 (0001)
  i=1: pre[1]= 1 (0001) ^ arr[1]= 3 (0011) -> pre[2]= 2 (0010)
  i=2: pre[2]= 2 (0010) ^ arr[2]= 4 (0100) -> pre[3]= 6 (0110)
  i=3: pre[3]= 6 (0110) ^ arr[3]= 8 (1000) -> pre[4]=14 (1110)
Pha 2: trả lời truy vấn
  [0,1]: pre[2]=2 ^ pre[0]=0 = 2
  [1,2]: pre[3]=6 ^ pre[1]=1 = 7
  [0,3]: pre[4]=14 ^ pre[0]=0 = 14
  [3,3]: pre[4]=14 ^ pre[3]=6 = 8
=> [2, 7, 14, 8]


[2, 7, 14, 8]

## 9. Code nộp LeetCode

Bản chính thức: prefix XOR với phần tử `0` đứng đầu (tránh mọi trường hợp đặc biệt khi `l = 0`).

In [20]:
class Solution:
    def xorQueries(self, arr: list[int], queries: list[list[int]]) -> list[int]:
        pre = [0] * (len(arr) + 1)          # pre[i] = arr[0] ^ ... ^ arr[i-1]
        for i, x in enumerate(arr):
            pre[i + 1] = pre[i] ^ x
        # arr[l] ^ ... ^ arr[r] = pre[r+1] ^ pre[l]  (phần chung tự huỷ vì a ^ a = 0)
        return [pre[r + 1] ^ pre[l] for l, r in queries]

sol = Solution()
for (arr, queries), want in examples:
    got = sol.xorQueries(arr, queries)
    print(got, "✔" if got == want else "✘")

[2, 7, 14, 8] ✔
[8, 0, 4, 4] ✔


## 10. Vì sao thuật toán đúng

1. **Dựng đúng:** quy nạp — `pre[0] = 0` là XOR của dãy rỗng; nếu `pre[i]` là XOR của `arr[0..i-1]` thì `pre[i+1] = pre[i] ^ arr[i]` là XOR của `arr[0..i]`.
2. **Truy vấn đúng:** nhờ tính kết hợp và giao hoán, `pre[r+1] = pre[l] ^ (arr[l] ^ … ^ arr[r])`. XOR hai vế với `pre[l]`: `pre[r+1] ^ pre[l] = pre[l] ^ pre[l] ^ (arr[l] ^ … ^ arr[r]) = 0 ^ (…) = arr[l] ^ … ^ arr[r]`.
3. Vì `0 <= l <= r < n` nên `r + 1 <= n` và `l >= 0` — chỉ số luôn hợp lệ.

Không có giả thiết gì về giá trị (âm/dương/lớn) — Python dùng số nguyên không giới hạn, và XOR của các số `<= 10^9 < 2^30` vẫn `< 2^30`.

## 11. Phân tích Time / Space

Gọi `n = len(arr)`, `q = len(queries)` (đều ≤ `3·10^4`).

| Cách | Tiền xử lý | Mỗi truy vấn | Tổng Time | Space phụ |
|---|---|---|---|---|
| 1. Brute force | – | `O(n)` | `O(q·n)` ≈ `9·10^8` | `O(1)` |
| 2. Chia khối | `O(n)` | `O(√n)` | `O(n + q√n)` | `O(√n)` |
| 3. Prefix XOR | `O(n)` | `O(1)` | `O(n + q)` | `O(n)` |
| 4. accumulate / tại chỗ | `O(n)` | `O(1)` | `O(n + q)` | `O(n)` / `O(1)` |

Danh sách kết quả luôn chiếm `O(q)` (không tính vào bộ nhớ phụ).

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, truy vấn một phần tử, truy vấn cả mảng, các phần tử bằng nhau cho kết quả 0, giá trị lớn `10^9`), và đối chiếu các cách với brute force trên input ngẫu nhiên có seed cố định.

In [21]:
ALL = [xor_queries_brute, xor_queries_blocks, xor_queries_prefix, xor_queries_accumulate,
       lambda a, q: xor_queries_inplace(list(a), q),
       Solution().xorQueries, SolutionBlocks().xorQueries, SolutionPrefix().xorQueries, SolutionAccumulate().xorQueries]
tests = [
    (([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]), [2, 7, 14, 8]),
    (([4, 8, 2, 10], [[2, 3], [1, 3], [0, 0], [0, 3]]), [8, 0, 4, 4]),
    (([7], [[0, 0]]), [7]),                                   # n = 1
    (([5, 5], [[0, 1], [0, 0], [1, 1]]), [0, 5, 5]),          # a ^ a = 0
    (([10**9, 10**9, 1], [[0, 2]]), [1]),                     # giá trị lớn
    (([1, 2, 3], [[0, 2], [0, 2]]), [0, 0]),                  # truy vấn lặp lại; 1^2^3 = 0
    (([2, 4, 8, 16], [[1, 2], [0, 3]]), [12, 30]),
]
for (arr, qs), want in tests:
    for f in ALL:
        got = f(arr, qs)
        assert got == want, (f, arr, qs, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 7 test cố định qua cho 9 hàm


In [22]:
rng = random.Random(1310)
for _ in range(500):
    n = rng.randint(1, 40)
    arr = [rng.randint(1, 10**9) for _ in range(n)]
    qs = []
    for _ in range(rng.randint(1, 30)):
        l = rng.randrange(n); r = rng.randrange(l, n)
        qs.append([l, r])
    want = xor_queries_brute(arr, qs)
    for f in ALL[1:]:
        assert f(arr, qs) == want
print("Đối chiếu 500 bộ ngẫu nhiên: khớp")
t0 = time.perf_counter()
Solution().xorQueries(arr_big, qs_big)
print(f"Solution trên n=q=3·10^4: {time.perf_counter() - t0:.4f}s")

Đối chiếu 500 bộ ngẫu nhiên: khớp
Solution trên n=q=3·10^4: 0.0061s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lệch chỉ số: `pre[r] ^ pre[l]`

In [23]:
def wrong_offset(arr, queries):
    pre = list(accumulate(arr, operator.xor, initial=0))
    return [pre[r] ^ pre[l] for l, r in queries]        # SAI: thiếu +1

arr, qs = [1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]
print("sai :", wrong_offset(arr, qs))
print("đúng:", Solution().xorQueries(arr, qs))

sai : [1, 3, 6, 0]
đúng: [2, 7, 14, 8]


**Vì sao sai:** `pre[r]` chỉ chứa tới `arr[r-1]` ⇒ mất phần tử `arr[r]`; truy vấn `[3,3]` ra `0`. **Sửa:** với `pre` có số 0 ở đầu, đoạn `[l, r]` là `pre[r+1] ^ pre[l]`. Mẹo kiểm tra: truy vấn một phần tử `[i, i]` phải ra `arr[i]`.

### Lỗi 2 — Dùng phép trừ như prefix sum

In [24]:
def wrong_minus(arr, queries):
    pre = list(accumulate(arr, operator.xor, initial=0))
    return [pre[r + 1] - pre[l] for l, r in queries]    # SAI: XOR không "trừ" được

arr, qs = [4, 8, 2, 10], [[2, 3], [1, 3], [0, 0], [0, 3]]
print("sai :", wrong_minus(arr, qs))
print("đúng:", Solution().xorQueries(arr, qs))

sai : [-8, 0, 4, 4]
đúng: [8, 0, 4, 4]


**Vì sao sai:** phép ngược của `^` là chính `^`, không phải `-`. Ví dụ `[2,3]`: `4 − 12 = −8` thay vì `4 ^ 12 = 8`. **Sửa:** `pre[r+1] ^ pre[l]`.

### Lỗi 3 — Tiền tố không có số 0 ở đầu và dùng `pre[l-1]` khi `l = 0`

In [25]:
def wrong_minus_one(arr, queries):
    pre = list(accumulate(arr, operator.xor))           # pre[i] = XOR arr[0..i]
    return [pre[r] ^ pre[l - 1] for l, r in queries]    # SAI khi l = 0: pre[-1] là phần tử CUỐI

arr, qs = [1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]
print("sai :", wrong_minus_one(arr, qs), "<- không có lỗi nào được báo!")
print("đúng:", Solution().xorQueries(arr, qs))

sai : [12, 7, 0, 8] <- không có lỗi nào được báo!
đúng: [2, 7, 14, 8]


**Vì sao sai:** trong Python `pre[-1]` hợp lệ (phần tử cuối) nên chương trình **không báo lỗi** mà âm thầm ra kết quả sai cho mọi truy vấn có `l = 0`. **Sửa:** thêm `0` vào đầu (`initial=0`) hoặc viết `pre[l-1] if l > 0 else 0`.

### Lỗi 4 — Nhầm `^` với luỹ thừa khi tự kiểm tra

Một số bạn viết thêm "kiểm tra" kiểu toán học, ví dụ đếm số bit bằng `2^k`.

In [26]:
def wrong_bits(x):
    # muốn đếm số bit 1 của x bằng cách thử từng 2^k
    return sum(1 for k in range(31) if x & (2 ^ k))     # SAI: 2 ^ k là XOR

def right_bits(x):
    return sum(1 for k in range(31) if x & (2 ** k))    # hoặc (1 << k), hoặc bin(x).count("1")

for x in [14, 8, 7]:
    print(x, "-> sai:", wrong_bits(x), "| đúng:", right_bits(x), "| bin.count:", bin(x).count("1"))

14 -> sai: 27 | đúng: 3 | bin.count: 3
8 -> sai: 15 | đúng: 1 | bin.count: 1
7 -> sai: 27 | đúng: 3 | bin.count: 3


**Vì sao sai:** `2 ^ k` là XOR (với `k = 2` cho `0`, với `k = 3` cho `1`, …), không phải `2^k`. **Sửa:** dùng `1 << k` hoặc `2 ** k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhiều truy vấn đoạn `[l, r]`, mảng không đổi** ⇒ nghĩ ngay **tiền tố** (prefix).
- Tiền tố dùng được khi phép toán **kết hợp** và **có nghịch đảo**: `+` (ngược `−`), `^` (ngược chính nó), `*` trên số khác 0 (ngược `/`, cẩn thận).
- **XOR + "đoạn con"** ⇒ `xor(l..r) = pre[r+1] ^ pre[l]`; **"đếm đoạn có XOR = k"** ⇒ prefix + hashmap.
- Luôn thêm phần tử `0` ở đầu mảng tiền tố để không phải xử lý riêng `l = 0`.
- Kiểm tra nhanh: truy vấn `[i, i]` phải cho `arr[i]`.
- Có **cập nhật** phần tử ⇒ Fenwick tree / segment tree với phép `^`.

## 15. Biến thể của bài

1. **LeetCode 303 — Range Sum Query Immutable**: cùng ý tưởng với phép cộng.
2. **LeetCode 1442 — Count Triplets That Can Form Two Arrays of Equal XOR**: `a == b` ⇔ XOR đoạn `[i, k]` bằng 0 ⇔ `pre[i] == pre[k+1]`.
3. **Đếm số đoạn con có XOR bằng `k`**: prefix XOR + `Counter` (bài tập 5).
4. **LeetCode 421 — Maximum XOR of Two Numbers**: trie nhị phân.
5. **Có cập nhật điểm**: Fenwick tree với `^` — vì `^` có nghịch đảo nên cập nhật `i` bằng `delta = old ^ new`.

In [27]:
# Biến thể 5: Fenwick tree (BIT) cho XOR với cập nhật điểm
class XorBIT:
    def __init__(self, arr):
        self.n = len(arr)
        self.t = [0] * (self.n + 1)
        self.a = list(arr)
        for i, x in enumerate(arr):
            self._add(i, x)
    def _add(self, i, delta):
        i += 1
        while i <= self.n:
            self.t[i] ^= delta
            i += i & -i
    def prefix(self, i):            # XOR của a[0..i-1]
        res = 0
        while i > 0:
            res ^= self.t[i]
            i -= i & -i
        return res
    def update(self, i, val):
        self._add(i, self.a[i] ^ val)
        self.a[i] = val
    def query(self, l, r):
        return self.prefix(r + 1) ^ self.prefix(l)

bit = XorBIT([1, 3, 4, 8])
print([bit.query(l, r) for l, r in [[0, 1], [1, 2], [0, 3], [3, 3]]])
bit.update(1, 6)                 # arr thành [1, 6, 4, 8]
print("sau cập nhật arr[1]=6:", bit.query(0, 3), "=", 1 ^ 6 ^ 4 ^ 8)

[2, 7, 14, 8]
sau cập nhật arr[1]=6: 11 = 11


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | XOR từng đoạn | `O(q·n)` | `O(1)` | Kiểm chứng, input nhỏ |
| 2. Chia khối | Khối `√n` tính sẵn | `O(n + q√n)` | `O(√n)` | Khi có cập nhật + muốn đơn giản |
| 3. Prefix XOR | `pre[r+1] ^ pre[l]` | `O(n + q)` | `O(n)` | Bản nộp chuẩn |
| 4. accumulate / tại chỗ | Như 3, code gọn / không mảng phụ | `O(n + q)` | `O(n)` / `O(1)` | Viết nhanh / tiết kiệm bộ nhớ |
| (Biến thể) Fenwick | Cây chỉ số nhị phân với `^` | `O((n + q) log n)` | `O(n)` | Có cập nhật điểm |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Giá trị của `5 ^ 5`, `5 ^ 0`, `1 ^ 2 ^ 3`, `2 ^ 3` trong Python?

**Bài 2 (tính tay):** Dựng mảng tiền tố XOR cho `arr = [6, 2, 7, 3]` và trả lời truy vấn `[1, 3]`.

**Bài 3 (điền TODO):** Hoàn thành `build_prefix_xor(arr)` trả về `pre` dài `n + 1`.

**Bài 4 (tự viết lại):** Viết lại `my_xor_queries(arr, queries)` không xem code.

**Bài 5 (mini challenge):** Viết `count_subarrays_xor_k(arr, k)`: đếm số đoạn con liên tiếp có XOR bằng `k`, trong `O(n)`.

In [28]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def build_prefix_xor_todo(arr):
    pre = [0] * (len(arr) + 1)
    # TODO: pre[i + 1] = pre[i] ^ arr[i]
    return pre

def my_xor_queries_todo(arr, queries):
    # TODO
    pass

def count_subarrays_xor_k_todo(arr, k):
    # TODO: duyệt tiền tố; số đoạn kết thúc tại i có XOR = k
    #       = số lần đã gặp giá trị tiền tố (cur ^ k)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `a ^ a = 0`. (2) `a ^ 0 = a`. (3) `1 ^ 2 = 3`, rồi `3 ^ 3 = 0`; `2 ^ 3` là XOR chứ không phải 8.

**Bài 2:** (1) `pre[0] = 0`. (2) `pre = [0, 6, 4, 3, 0]`. (3) `[1,3]` = `pre[4] ^ pre[1] = 0 ^ 6 = 6` (kiểm tra: `2 ^ 7 ^ 3 = 6`).

**Bài 3:** (1) Vòng `for i, x in enumerate(arr)`. (2) `pre[i+1] = pre[i] ^ x`. (3) Trả `pre`.

**Bài 4:** (1) Dựng `pre`. (2) Duyệt từng `[l, r]`. (3) `pre[r+1] ^ pre[l]`.

**Bài 5:** (1) Đoạn `(j, i]` có XOR `k` ⇔ `pre[i] ^ pre[j] = k` ⇔ `pre[j] = pre[i] ^ k`. (2) Dùng `Counter` đếm các giá trị tiền tố đã gặp, khởi tạo `{0: 1}`. (3) Với mỗi `i`: `ans += cnt[cur ^ k]`, rồi `cnt[cur] += 1`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
from collections import Counter

# Bài 1
assert (5 ^ 5, 5 ^ 0, 1 ^ 2 ^ 3, 2 ^ 3) == (0, 5, 0, 1)

# Bài 2
pre2 = list(accumulate([6, 2, 7, 3], operator.xor, initial=0))
assert pre2 == [0, 6, 4, 3, 0]
assert pre2[4] ^ pre2[1] == 2 ^ 7 ^ 3 == 6

# Bài 3
def build_prefix_xor(arr):
    pre = [0] * (len(arr) + 1)
    for i, x in enumerate(arr):
        pre[i + 1] = pre[i] ^ x
    return pre

assert build_prefix_xor([1, 3, 4, 8]) == [0, 1, 2, 6, 14]
assert build_prefix_xor([]) == [0]
print("Bài 1–3: OK")

Bài 1–3: OK


In [30]:
# Bài 4
def my_xor_queries(arr, queries):
    pre = build_prefix_xor(arr)
    return [pre[r + 1] ^ pre[l] for l, r in queries]

# Bài 5
def count_subarrays_xor_k(arr, k):
    cnt = Counter({0: 1})
    cur = ans = 0
    for x in arr:
        cur ^= x
        ans += cnt[cur ^ k]
        cnt[cur] += 1
    return ans

def count_brute(arr, k):
    n = len(arr)
    return sum(1 for l in range(n) for r in range(l, n) if reduce(operator.xor, arr[l:r + 1], 0) == k)

assert count_subarrays_xor_k([4, 2, 2, 6, 4], 6) == 4
rng = random.Random(77)
for _ in range(400):
    n = rng.randint(1, 15)
    arr = [rng.randint(0, 7) for _ in range(n)]
    k = rng.randint(0, 7)
    assert count_subarrays_xor_k(arr, k) == count_brute(arr, k)
    qs = [[l, r] for l in range(n) for r in range(l, n)]
    assert my_xor_queries(arr, qs) == xor_queries_brute(arr, qs)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tiền tố cho phép toán có nghịch đảo:**

```python
pre = [IDENTITY] * (n + 1)              # tổng: 0, XOR: 0
for i, x in enumerate(arr):
    pre[i + 1] = OP(pre[i], x)          # tổng: +, XOR: ^
# đoạn [l, r]:
ans = INV(pre[r + 1], pre[l])           # tổng: pre[r+1] - pre[l], XOR: pre[r+1] ^ pre[l]
```

**Bài 1310 trong 3 dòng:**

```python
pre = list(accumulate(arr, operator.xor, initial=0))
return [pre[r + 1] ^ pre[l] for l, r in queries]
```

**Checklist trước khi submit:**
- [ ] Mảng tiền tố dài `n + 1`, có `0` ở đầu.
- [ ] Đoạn `[l, r]` dùng `pre[r+1]` và `pre[l]` (kiểm tra bằng `[i, i] → arr[i]`).
- [ ] Dùng `^`, không dùng `-`.
- [ ] Không dùng chỉ số `-1` "vô tình".
- [ ] Đúng chữ ký: `xorQueries(self, arr: list[int], queries: list[list[int]]) -> list[int]`.